# DeiT-Tiny seed 4: E→O raw top1 vs relaxed GELU Bypass

Attach CIFAR-100, enable Internet, T4x2 and Kaggle Secret `github_token`.
Optional: full previous output archive from this exact seed/config for resume.
This seed trains its own Vanilla and must not reuse another seed's fork.

Phase1: train Vanilla until150 epochs without a strict validation-accuracy best.
No minimum epoch. Reload the historical-best model/AdamW/scheduler/RNG/loader.
Phase2: GPU0 E→O raw top1, GPU1 `deit_bypass`;150 training epochs each.
No A3/A4 or additional arm. Original raw TINY WHERE over12 sites, rank4,
WHERE3x32, projection32, gate32, adaptive CG200, scales(.0125,.025,.05).
E→O uses patience10 rollback (accuracy/lower loss tie anchor), current RNG/data
stream, and no retrigger. Report best uses strict accuracy with paired loss.

Bypass follows Jung/Lee's relaxed construction (Eq.8 / Section IV-A):
`GELU(z) + d ⊙ z` at all12 original MLP activations, initialized at d=0.
It trains all original weights and D: opt1=100 epochs; opt2 has at most50 epochs,
with task loss + `(3e-6 * opt2_step) * sum(||d||₂)`.
When summed contraction norm <.002, drop D and spend remaining epochs in train3.
No forced projection, rollback or scheduler rebase in Bypass. The existing
AdamW moments/groups are preserved; new D vectors join the no-decay group.
This is a GELU adaptation with a matched budget, not a reproduction of the
paper's original network/hyperparameter schedule.

Only original-space validation observations after contraction qualify as Bypass
accuracy comparisons. Expanded-space best is a separate diagnostic. If opt2
exhausts the budget before contraction, `accuracy_comparison_eligible=false`
and the comparison delta is null. Validation never controls contraction.

Recipe: batch64, AdamW LR5e-4/WD.05, warmup5, cosine schedule300, max_epoch800,
validation3000, plateau150 and post-fork150. Schedule300 is not a minimum epoch.
Full latest saves every epoch, including Bypass phase/counters/D/optimizer/RNG,
or E controller/anchor. Completed arms skip training and partial arms resume.
Outputs: result/checkpoints/epoch logs and bypass_comparison.csv/json. Download
this notebook's full output tar.gz for the next Kaggle session.
Paper: https://www.donghunlee.com/papers/Jung_Lee_Bypass__IEEE_TNNLS.pdf


In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO = Path("/kaggle/working/deit-all-arms-repo")
OUTPUT = Path("/kaggle/working/projection_log_analysis")
URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
token = UserSecretsClient().get_secret("github_token").strip()
if not token:
    raise RuntimeError("Enable the github_token Kaggle Secret")
askpass = Path("/kaggle/working/.projection_log_git_askpass.py")
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
try:
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", URL, str(REPO)], env=env, check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "main"], env=env, check=True)
        subprocess.run(["git", "-C", str(REPO), "merge", "--ff-only", "origin/main"], env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop("GITHUB_TOKEN_RUNTIME", None)
    del token
GROMO = Path("/kaggle/working/deit-gromo")
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
if not GROMO.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                    "https://github.com/growingnet/gromo.git", str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1", "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach", GROMO_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[vision,test]", "-e", str(GROMO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
ENV = os.environ.copy()
ENV["PYTHONPATH"] = os.pathsep.join((str(REPO), str(GROMO / "src")))
ENV["REQUIRE_DEIT_INTEGRATION"] = "1"
subprocess.run([sys.executable, "-m", "pytest", "-q",
    "tests/test_deit_function_preserving.py", "tests/test_deit_sites.py",
    "tests/test_deit_projection.py", "tests/test_deit_no_aux_persistence.py",
    "tests/test_deit_protocol.py", "tests/test_deit_vanilla_reference.py", "tests/test_deit_e_rollback.py", "tests/test_deit_ablation.py", "tests/test_deit_ablation_integration.py", "tests/test_deit_logging.py", "tests/test_deit_parallel.py", "tests/test_deit_resume.py", "tests/test_deit_bypass.py", "tests/test_kaggle_checkpoint_discovery.py"], cwd=REPO, env=ENV, check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable a CUDA GPU; this notebook trains DeiT")
print("Repository commit:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
from dataclasses import asdict
from experiments.deit_protocol import DeitRecipe, protocol, canonical_model_config
from adapters.deit_ablation import res18_cp_config
from experiments.shared_protocol import sha256_file

SEED = 4
ARMS = ('e_driven_o_raw', 'deit_bypass')
SCHEDULE_EPOCHS = 300
STALL_PATIENCE = 150
POST_FORK_EPOCHS = 150
ALGORITHM_PATIENCE = 10
MAX_EPOCH = 800
BATCH_SIZE = 64
GPU_DEVICES = 'auto'  # T4x2: two workers; one GPU: one worker.
LEARNING_RATE = 5e-4  # Explicit per-arm LR; no implicit ImageNet batch scaling.
RANK = 4
PROJECTION_SAMPLES = 32
SCALES = '.0125,.025,.05'
OUTPUT = Path(f'/kaggle/working/deit_tiny_seed{SEED}_raw_vs_bypass_v1')
OUTPUT.mkdir(parents=True, exist_ok=True)
RECIPE = DeitRecipe(seed=SEED, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    schedule_epochs=SCHEDULE_EPOCHS, stall_start_epoch=0,
    stall_patience=STALL_PATIENCE, reference_epochs=POST_FORK_EPOCHS, max_epoch=MAX_EPOCH)
RECIPE.validate()
EXPECTED = protocol(RECIPE, canonical_model_config())
CP = res18_cp_config()
from dataclasses import replace
CP = replace(CP, rank=RANK, projection_samples=PROJECTION_SAMPLES,
             scales=tuple(map(float, SCALES.split(','))))
roots = set()
for directory, _, files in os.walk('/kaggle/input', followlinks=True):
    if Path(directory).name == 'cifar-100-python' and {'train', 'meta'}.issubset(files):
        roots.add(Path(directory).parent.resolve())
if len(roots) != 1:
    raise FileNotFoundError(f'Attach exactly one CIFAR-100 dataset: {roots}')
DATA_ROOT = next(iter(roots))
print('Recipe:', asdict(RECIPE))
print('Projection:', asdict(CP))

BYPASS_OPT1_EPOCHS = 100
BYPASS_MAX_OPT2_EPOCHS = 50


In [ ]:
# Import full outputs (original .pt, .tar.gz/.zip, or mounted notebook output).
import shutil
from experiments.deit_resume import prepare_resume
RESUME_PLAN = prepare_resume(['/kaggle/input'], OUTPUT, RECIPE, CP,
    horizon=POST_FORK_EPOCHS, patience=ALGORITHM_PATIENCE, inner_steps=5, arms=ARMS)
PHASE1 = OUTPUT / 'vanilla_stall'
FORK = PHASE1 / 'plateau_checkpoint.pt'
VANILLA_REFERENCE = PHASE1 / 'vanilla_reference.pt'


In [ ]:
command = [sys.executable, '-m', 'experiments.run_deit_all_arms', '--data-root', str(DATA_ROOT),
    '--output', str(OUTPUT), '--seed', str(SEED), '--schedule-epochs', str(SCHEDULE_EPOCHS),
    '--max-epoch', str(MAX_EPOCH), '--batch-size', str(BATCH_SIZE), '--learning-rate', str(LEARNING_RATE),
    '--stall-patience', str(STALL_PATIENCE), '--post-fork-epochs', str(POST_FORK_EPOCHS),
    '--algorithm-patience', str(ALGORITHM_PATIENCE), '--rank', str(RANK),
    '--projection-samples', str(PROJECTION_SAMPLES), '--scales', SCALES, '--opt-inner-steps', '5',
    '--gpu-devices', GPU_DEVICES, '--arms', ','.join(ARMS),
'--bypass-opt1-epochs', str(BYPASS_OPT1_EPOCHS), '--bypass-max-opt2-epochs', str(BYPASS_MAX_OPT2_EPOCHS)]
if FORK.exists():
    command += ['--plateau-checkpoint', str(FORK), '--vanilla-reference', str(VANILLA_REFERENCE)]
try:
    subprocess.run(command, cwd=REPO, env=ENV, check=True)
finally:
    archive = shutil.make_archive(str(OUTPUT), 'gztar', root_dir=OUTPUT)
    print('Resumable output archive:', archive)


In [ ]:
path = OUTPUT / 'bypass_comparison.json'
if path.exists():
    print(json.dumps(json.loads(path.read_text()), indent=2))
print((OUTPUT / 'arm_status.json').read_text())
